In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from tqdm.notebook import tqdm
import optuna
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

Path("results").mkdir(exist_ok=True)

print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}")

In [ ]:
from src.data.data_loader import get_dataset
from src.evaluation.experiments import run_link_prediction_experiment

In [ ]:
datasets_to_load = ['cora', 'pubmed', 'facebook', 'twitch_ru', 'lastfm_asia', 'ogbn_arxiv']

graphs = {}
for name in datasets_to_load:
    ds = get_dataset(name, verbose=True)
    graphs[name] = ds

In [ ]:
lp_datasets = ['cora', 'pubmed', 'facebook', 'twitch_ru', 'lastfm_asia', 'ogbn_arxiv']

lp_results = []

for ds_name in tqdm(lp_datasets, desc="Datasets"):
    print(f"\n{'='*70}\n🚀 Link Prediction → {ds_name.upper()}\n{'='*70}")
    try:
        res = run_link_prediction_experiment(
            ds_name,
            use_features=True,
            n_runs=3,
            force_recompute=False
        )
        lp_results.append(res)
        tqdm.write(f"OK {ds_name}: AUC={res.get('auc_mean', 0):.4f}, AP={res.get('ap_mean', 0):.4f}")
    except Exception as e:
        tqdm.write(f"FAIL {ds_name}: {repr(e)}")

In [ ]:
# Секция 5 – Link Prediction (полный прогон)
from src.evaluation.experiments import run_link_prediction_experiment

lp_datasets = ['cora', 'pubmed', 'facebook', 'twitch_ru', 'lastfm_asia', 'ogbn_arxiv']
lp_models = ['logistic','mlp','gcn','graphsage','gat','gatv2',
             'graph_transformer','gin','sgc','jknet','gcnii']
lp_results = []

for ds_name in tqdm(lp_datasets, desc="LP Datasets"):
    ds = graphs[ds_name]
    for model in tqdm(lp_models, desc=f"LP {ds_name}", leave=False):
        if ds_name == 'ogbn_arxiv' and model in ['logistic', 'mlp']:
            continue
        try:
            res = run_link_prediction_experiment(
                ds_name, method=model, use_features=True, use_embeddings=False,
                dimensions=64, n_runs=2, force_recompute=False
            )
            lp_results.append(res)
            tqdm.write(f"OK {ds_name} {model} feat: AUC={res['auc_mean']:.4f}")
        except Exception as e:
            tqdm.write(f"FAIL {ds_name} {model}: {repr(e)}")

df_lp = pd.DataFrame(lp_results)
df_lp.to_csv('results/link_prediction_full.csv', index=False)
df_lp

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import networkx as nx

HEURISTICS = ['common_neighbors', 'jaccard', 'adamic_adar', 'preferential_attachment', 'resource_allocation']
lp_heuristic_results = []

for ds_name in tqdm(lp_datasets, desc="Heuristics"):
    G = graphs[ds_name]['graph']
    G_u = G.to_undirected() if G.is_directed() else G
    from src.models.data import make_link_prediction_data
    data_dict = make_link_prediction_data(G, test_ratio=0.1, val_ratio=0.05, random_state=42)
    train_pos = data_dict['train_pos'].numpy().T
    test_pos = data_dict['test_pos'].numpy().T
    test_neg = data_dict['test_neg'].numpy().T
    
    idx_to_node = {v: k for k, v in data_dict['node_to_idx'].items()}
    train_pos_orig = [(idx_to_node[u], idx_to_node[v]) for u, v in train_pos]
    test_pos_orig = [(idx_to_node[u], idx_to_node[v]) for u, v in test_pos]
    test_neg_orig = [(idx_to_node[u], idx_to_node[v]) for u, v in test_neg]
    
    G_train = nx.Graph()
    G_train.add_nodes_from(G_u.nodes())
    G_train.add_edges_from(train_pos_orig)
    
    for heur in HEURISTICS:
        try:
            scores, y_true = [], []
            for pairs, label in [(test_pos_orig, 1), (test_neg_orig, 0)]:
                for u, v in pairs:
                    if heur == 'common_neighbors':
                        s = len(list(nx.common_neighbors(G_train, u, v)))
                    elif heur == 'jaccard':
                        cn = len(list(nx.common_neighbors(G_train, u, v)))
                        nb_u = set(G_train.neighbors(u))
                        nb_v = set(G_train.neighbors(v))
                        union = len(nb_u | nb_v)
                        s = cn / union if union > 0 else 0
                    elif heur == 'adamic_adar':
                        s = sum(1 / np.log(G_train.degree(w) + 1e-10) for w in nx.common_neighbors(G_train, u, v))
                    elif heur == 'preferential_attachment':
                        s = G_train.degree(u) * G_train.degree(v)
                    elif heur == 'resource_allocation':
                        s = sum(1 / (G_train.degree(w) + 1e-10) for w in nx.common_neighbors(G_train, u, v))
                    else:
                        s = 0
                    scores.append(s)
                    y_true.append(label)
            auc = roc_auc_score(y_true, scores)
            ap = average_precision_score(y_true, scores)
            lp_heuristic_results.append({
                'dataset': ds_name,
                'method': f'heuristic_{heur}',
                'auc': auc,
                'ap': ap
            })
            tqdm.write(f"OK {ds_name} {heur}: AUC={auc:.4f}, AP={ap:.4f}")
        except Exception as e:
            tqdm.write(f"FAIL {ds_name} {heur}: {e}")

df_heur = pd.DataFrame(lp_heuristic_results)
df_heur.to_csv('results/link_prediction_heuristics.csv', index=False)
df_heur

In [ ]:
# Находим лучшую GNN-модель для каждого датасета
best_gnn_lp = {}

for ds_name in lp_datasets:
    ds_res = [r for r in lp_results if r.get('dataset') == ds_name]
    gnn_res = [r for r in ds_res if r.get('model') in ['graphsage', 'gat', 'gatv2', 'gin', 'gcn']]
    if gnn_res:
        best = max(gnn_res, key=lambda x: x.get('auc_mean', 0))
        best_gnn_lp[ds_name] = best['model']
        print(f"Best GNN for {ds_name}: {best['model']} (AUC = {best.get('auc_mean', 0):.4f})")

hyper_lp_results = []

for ds_name, best_model in best_gnn_lp.items():
    print(f"\nOptuna tuning for {ds_name} → {best_model} (20 trials)")

    def objective(trial):
        hidden_dim = trial.suggest_int("hidden_dim", 64, 256, step=32)
        num_layers = trial.suggest_int("num_layers", 1, 3)
        dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.05)

        res = run_link_prediction_experiment(
            ds_name,
            model=best_model,
            use_features=True,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            n_runs=2,
            force_recompute=False
        )
        return res.get('auc_mean', 0.0)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=20, show_progress_bar=True)

    best_params = study.best_params
    best_value = study.best_value

    # Финальный запуск с лучшими параметрами
    final_res = run_link_prediction_experiment(
        ds_name,
        model=best_model,
        use_features=True,
        hidden_dim=best_params["hidden_dim"],
        num_layers=best_params["num_layers"],
        dropout=best_params["dropout"],
        n_runs=3,
        force_recompute=False
    )

    hyper_lp_results.append({
        "dataset": ds_name,
        "best_model": best_model,
        "hidden_dim": best_params["hidden_dim"],
        "num_layers": best_params["num_layers"],
        "dropout": best_params["dropout"],
        "best_trial_auc": best_value,
        "final_auc": final_res.get('auc_mean', 0),
        "final_ap": final_res.get('ap_mean', 0)
    })

    print(f"   Best params: {best_params} | Final AUC = {final_res.get('auc_mean', 0):.4f}")

In [ ]:
df_lp = pd.DataFrame(lp_results)
df_lp_hyper = pd.DataFrame(hyper_lp_results)

df_lp.to_csv('results/link_prediction_results.csv', index=False)
df_lp_hyper.to_csv('results/link_prediction_hyper_tuning.csv', index=False)

print("\n=== Базовые результаты Link Prediction ===")
display(df_lp)

print("\n=== Результаты Optuna-тюнинга лучших GNN ===")
display(df_lp_hyper)